# Risk Parameters for BTC

In [1]:
import os
from typing import Tuple
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import sys
sys.path.append(os.getcwd().split("scripts")[0])
sys.path.append(os.path.join(os.getcwd().split("scripts")[0], "params"))

import params
from params import funding, caps
import pystable

from scipy.stats import levy_stable

In [2]:
filename = "btc"

path_to_file = os.path.join(os.getcwd().split("scripts")[0], f"data/{filename}")

periodicity = 60. # 1 minute in seconds
cap = 10  # cap on pay off, set by governance

short_twap = 10 * periodicity
long_twap = 60 * periodicity

periodicity, short_twap, long_twap

(60.0, 600.0, 3600.0)

In [3]:
df = pd.read_csv(path_to_file+".csv", parse_dates=["timestamp"]).set_index("timestamp")
df

,close
timestamp,
2024-01-01 00:01:00+00:00,42298.61
2024-01-01 00:02:00+00:00,42320.00
2024-01-01 00:03:00+00:00,42325.50
2024-01-01 00:04:00+00:00,42367.99
2024-01-01 00:05:00+00:00,42397.23
...,...
2024-05-08 13:06:00+00:00,62097.67
2024-05-08 13:07:00+00:00,62077.49
2024-05-08 13:08:00+00:00,62115.51


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 185110 entries, 2024-01-01 00:01:00+00:00 to 2024-05-08 13:10:00+00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   close   185110 non-null  float64
dtypes: float64(1)
memory usage: 2.8 MB


Computing the 1h TWAP, sampled every 10 min.

In [5]:
df_twap = df.rolling(60).mean().dropna().resample('10min').last()
df_twap

,close
timestamp,
2024-01-01 01:00:00+00:00,42445.579667
2024-01-01 01:10:00+00:00,42443.685833
2024-01-01 01:20:00+00:00,42450.079333
2024-01-01 01:30:00+00:00,42485.129333
2024-01-01 01:40:00+00:00,42528.335833
...,...
2024-05-08 12:30:00+00:00,62331.672000
2024-05-08 12:40:00+00:00,62302.007667
2024-05-08 12:50:00+00:00,62279.337333


In [27]:
df_twap.plot()

Looking at the indexes

In [7]:
df.index.is_unique, df.index.is_monotonic_increasing

(True, True)

In [8]:
diff = np.diff(df.index.to_numpy())
print(f"Is index equally spaced? {'YES' if np.all(diff == diff[0]) else 'NO'} ")

Is index equally spaced? YES 


In [10]:
def compute_log_return(_df:pd.DataFrame, column:str) -> None:
    if not column in _df.columns:
        raise Exception(f"Column {column} not found")
    _df["log_return"] = np.nan
    _df.loc[_df.index[1]:,"log_return"] = np.log(_df[column].iloc[1:].to_numpy() / _df[column].iloc[:-1].to_numpy())
    _df.dropna(inplace=True)

In [11]:
compute_log_return(df_twap, column="close")
df_twap

,close,log_return
timestamp,,
2024-01-01 01:10:00+00:00,42443.685833,-0.000045
2024-01-01 01:20:00+00:00,42450.079333,0.000151
2024-01-01 01:30:00+00:00,42485.129333,0.000825
2024-01-01 01:40:00+00:00,42528.335833,0.001016
2024-01-01 01:50:00+00:00,42562.376667,0.000800
...,...,...
2024-05-08 12:30:00+00:00,62331.672000,0.000106
2024-05-08 12:40:00+00:00,62302.007667,-0.000476
2024-05-08 12:50:00+00:00,62279.337333,-0.000364


In [12]:
compute_log_return(df, column="close")
df

,close,log_return
timestamp,,
2024-01-01 00:02:00+00:00,42320.00,0.000506
2024-01-01 00:03:00+00:00,42325.50,0.000130
2024-01-01 00:04:00+00:00,42367.99,0.001003
2024-01-01 00:05:00+00:00,42397.23,0.000690
2024-01-01 00:06:00+00:00,42409.20,0.000282
...,...,...
2024-05-08 13:06:00+00:00,62097.67,-0.000543
2024-05-08 13:07:00+00:00,62077.49,-0.000325
2024-05-08 13:08:00+00:00,62115.51,0.000612


### Fit Distribution

In [13]:
def fit_distribution(
    _df:pd.DataFrame, column:str, period:float, verbose:bool=False
) -> Tuple[pystable.STABLE_DIST, pystable.STABLE_DIST]:

    if not column in _df.columns:
        raise Exception(f"Column {column} not found")

    dst = funding.gaussian()
    pystable.fit(dst, _df[column].to_numpy(), _df.index.size)

    if verbose:
        print(f'''
            alpha: {dst.contents.alpha}, beta: {dst.contents.beta},
            mu: {dst.contents.mu_1}, sigma: {dst.contents.sigma}
            '''
        )

    scaled_dst = funding.rescale(dst, 1./period)
    scaled_dst_2 = caps.rescale(dst, 1./period)

    if verbose:
        print(f'''
            rescaled params (1/t = {1./period}):
            alpha: {scaled_dst.contents.alpha}, beta: {scaled_dst.contents.beta},
            mu: {scaled_dst.contents.mu_1}, sigma: {scaled_dst.contents.sigma}
            '''
        )
        print(f'''
            rescaled params (1/t = {1./period}):
            alpha: {scaled_dst_2.contents.alpha}, beta: {scaled_dst_2.contents.beta},
            mu: {scaled_dst_2.contents.mu_1}, sigma: {scaled_dst_2.contents.sigma}
            '''
        )

    return dst, scaled_dst_2

In [14]:
dst, scaled_dst = fit_distribution(
    df, "log_return", period=periodicity, verbose=True
)


            alpha: 1.4398185687536873, beta: 0.03143063846574884,
            mu: 1.035529684203383e-05, sigma: 0.0003567229771301606
            

            rescaled params (1/t = 0.016666666666666666):
            alpha: 1.4398185687536873, beta: 0.03143063846574884,
            mu: 1.7258828070056382e-07, sigma: 2.07657789339447e-05
            

            rescaled params (1/t = 0.016666666666666666):
            alpha: 1.4398185687536873, beta: 0.03143063846574884,
            mu: 1.7258828070056382e-07, sigma: 2.6748413659098222e-05
            


In [15]:
dst_twap, scaled_dst_twap = fit_distribution(
    df_twap, "log_return", period=10*periodicity, verbose=True
)


            alpha: 1.4354662004526826, beta: 0.04331015377551361,
            mu: 3.3278282724470275e-05, sigma: 0.00042640455678843444
            

            rescaled params (1/t = 0.0016666666666666668):
            alpha: 1.4354662004526826, beta: 0.04331015377551361,
            mu: 5.5463804540783794e-08, sigma: 4.948307645621189e-06
            

            rescaled params (1/t = 0.0016666666666666668):
            alpha: 1.4354662004526826, beta: 0.04331015377551361,
            mu: 5.5463804540783794e-08, sigma: 6.365374044690427e-06
            


## Compare Distribution and Data

In [16]:
def compare_dist_and_data(
    _df:pd.DataFrame,
    _column:str,
    _any_dst:pystable.STABLE_DIST,
    _bins:int=1000
) -> pd.DataFrame:

    #levy_stable.fit(df["log_return"].to_numpy())
    #levy_any_dst_rv = levy_stable(
    #    _any_dst.contents.alpha, _any_dst.contents.beta ,_any_dst.contents.mu_1, _any_dst.contents.sigma
    #)
    min_1pct = pystable.q(_any_dst, [0.01], 1)[0]
    max_99pct = pystable.q(_any_dst, [0.99], 1)[0]

    x = np.linspace(min_1pct, max_99pct, _bins)

    df_any_dst = pd.DataFrame(
        pystable.pdf(_any_dst, x, len(x)), columns=["dist_pdf"], index=x
    )

    df_any_dst["dist_cdf"] = pystable.cdf(_any_dst, x, len(x))

    pdf, bin_edges = np.histogram(
        _df[_column].to_numpy(), bins=_bins, density=True
    )

    df_any_dst["data_pdf"] = pdf
    df_any_dst["data_cdf"] = np.cumsum(pdf * np.diff(bin_edges))

    #df_any_dst[["unscaled_cdf", "data_cdf"]].plot()

    return df_any_dst

In [22]:
df_dst_dist = compare_dist_and_data(df, "log_return", dst)

df_scaled_dist = compare_dist_and_data(df, "log_return", scaled_dst)

df_twap_dst_dist = compare_dist_and_data(df_twap, "log_return", dst_twap)

df_twap_scaled_dist = compare_dist_and_data(df_twap, "log_return", scaled_dst_twap)

In [23]:
df_dst_dist[["dist_cdf", "data_cdf"]].plot()

In [24]:
df_dst_dist[["dist_pdf", "data_pdf"]].plot()

In [25]:
df_twap_dst_dist[["dist_cdf", "data_cdf"]].plot()

In [26]:
df_twap_dst_dist[["dist_pdf", "data_pdf"]].plot()